# 06 — Triage candidates and data quality

Suggested categories (CLAUDE.md §8): **1** no flags · **2** clarification (missing or inconsistent information,
"not aware" declarations, awareness gaps) · **3** technical assessment (unusual requirements: long wall time,
capability-size jobs, memory, GPU-only work, large scratch). These are *suggestions* to help the committee
decide where to look first. Unusual requirements call for proportionate evidence; the flags are not findings
against anyone.

*Principles: P1, P12 in the [analysis framework](https://github.com/svaradh/hpc-questionnaire/blob/main/docs/analysis-framework.md). Nothing here scores or ranks groups; flags are candidates for the committee to look at. Totals cover responding groups only, and "Don't know" is never counted as zero; a yes/no flag shown as `<NA>` means its inputs are unknown. Only the respondents' answers and the cluster configuration are used — nothing depends on which code a workload runs.*

**To run:** paste the Spreadsheet ID in the first code cell, then *Runtime → Run all*. Colab asks once per session to sign in with your IISER Google account.

In [ ]:
SPREADSHEET_ID = 'PASTE_YOUR_SPREADSHEET_ID_HERE'   # <- the long string in the sheet URL
REPO_REF = 'main'                                   # git branch/tag for the helper code and cluster config

import os, urllib.request
if not os.path.exists('hpc_analysis.py'):            # in Colab: fetch the helper modules once per session
    for f in ['sheets_client.py', 'hpc_analysis.py', 'hpc_plots.py', 'code_capabilities.json']:
        urllib.request.urlretrieve(f'https://raw.githubusercontent.com/svaradh/hpc-questionnaire/{REPO_REF}/analysis/{f}', f)

import pandas as pd
from sheets_client import load_data, load_config
import hpc_analysis as ha
import hpc_plots as hp
pd.set_option('display.max_colwidth', 120)

dfs = load_data(SPREADSHEET_ID)
cfg = load_config(ref=REPO_REF)
wl = ha.workloads(dfs, cfg)
cov = ha.coverage(dfs)
print(f"{cov['groups']} groups responded; {cov['workloads']} workloads described "
      f"({cov['groups_with_workloads']} groups with at least one).")

In [ ]:
tr = ha.triage(dfs, wl, cfg)
hp.counts(tr['suggested_category'].map({1: '1 — no flags', 2: '2 — clarification', 3: '3 — technical assessment'}),
          'Submissions by suggested category', 'Submissions',
          order=['1 — no flags', '2 — clarification', '3 — technical assessment'])
for _, r in tr.iterrows():
    print(f"{r['submission_id']}  {r['pi_name']}  — suggested category {r['suggested_category']}")
    print('    ' + (r['reasons'].replace(chr(10), chr(10) + '    ') if r['reasons'] else '(no flags)'))

## Answers to confirm (internal consistency)

Answers within one submission that do not fit together — e.g. more cores than the stated nodes hold, a sequence's
running time longer than its calendar time, measured scaling runs that differ from the reported doubling effect. They are
compared only with each other (and with node sizes from the cluster configuration), after submission; "don't know" and
blank answers never trigger a check. Each is a question for the respondent, not a finding; they also appear above as
category-2 reasons. Checking answers against outside knowledge of a code's performance is done by hand.

In [ ]:
ha.consistency_checks(dfs, wl, cfg)

## Why the Section D panels opened

Each panel is asked once per entry, whatever opened it; `opened_by` in each panel tab lists every trigger that held
(e.g. `walltime_over_limit; long_calc`). Counts per panel and trigger:

In [ ]:
ha.panel_reasons(dfs).pivot_table(index='trigger', columns='panel', values='entries', fill_value=0)

## Data quality: how many key numbers are unknown

In [ ]:
key = ['cores', 'walltime_h', 'jobs_year', 'jobs_year_unconstrained', 'concurrent', 'mem_node_gb', 'scratch_write_gb']
raw = dfs['Workloads']
pd.DataFrame({'answered': [(ha.num(raw[c]).notna()).sum() for c in key],
              "don't know": [ha.is_unknown(raw[c]).sum() for c in key],
              'blank': [(raw[c].astype(str).str.strip() == '').sum() for c in key]}, index=key)